# Goal of notebook is to compare two coco json files and do analysis on matches

In [37]:
import json
import numpy as np
import pandas as pd
from collections import defaultdict
from scipy.optimize import linear_sum_assignment
from pycocotools import mask as maskUtils
import cv2
import matplotlib.pyplot as plt

In [38]:
# ============================================================
# CONFIG 
GT_JSON   = "/home/mellanie/Desktop/Tristan_2026_code/detectron2/model2.5_data/test.json"
PRED_JSON = "/home/mellanie/Desktop/Tristan_2026_code/outputs/fusion2_output_100%_col0_yes_tangles.json"

IOU_THRESHOLD = 0.50 # use 0.50 for standard COCO; 0.35 suits thin root-hair structures
OUTPUT_CSV = "analysis/csv_data_for_analysis/fusion2_100%_yes_tangles.csv"
IS_FUSED = True

In [39]:
# ============================================================
# STEP 1: Load both JSONs
# ============================================================

with open(GT_JSON) as f: 
    gt_coco = json.load(f)

with open(PRED_JSON) as f: 
    pred_coco = json.load(f)

print(f"GT   images: {len(gt_coco['images'])},  annotations: {len(gt_coco['annotations'])}")
print(f"Pred images: {len(pred_coco['images'])}, annotations: {len(pred_coco['annotations'])}")

GT   images: 54,  annotations: 2885
Pred images: 54, annotations: 2668


In [40]:
# ============================================================
# STEP 2: Build lookup tables keyed on file_name
#
# All joining is done via file_name so mismatched image_ids
# between the two JSONs don't matter.
# Ecotype, height, width always come from the GT side.
# ============================================================

# get image metadata from ground truth
image_meta = {
    img["file_name"]: {
        "gt_image_id":   img["id"],
        "height":        img["height"],
        "width":         img["width"],
        "ecotype":       img.get("ecotype", img["file_name"]),
    }
    for img in gt_coco["images"]
}

# build a map for image id and filename in ground truth
gt_id_to_fname = {img["id"]: img["file_name"] for img in gt_coco["images"]}

# build a map for image id and filename in ground truth
pred_id_to_fname = {img["id"]: img["file_name"] for img in pred_coco["images"]}

# Build annotation lists keyed by file_name
gt_anns_by_fname   = defaultdict(list)
pred_anns_by_fname = defaultdict(list)

for ann in gt_coco["annotations"]:
    fname = gt_id_to_fname.get(ann["image_id"])
    if fname:
        gt_anns_by_fname[fname].append(ann)

for ann in pred_coco["annotations"]:
    fname = pred_id_to_fname.get(ann["image_id"])
    if fname:
        pred_anns_by_fname[fname].append(ann)

# Evaluate only images present in both JSONs
gt_fnames   = set(image_meta.keys())
pred_fnames = set(pred_id_to_fname.values())
all_fnames  = sorted(gt_fnames & pred_fnames)

missing_in_pred = gt_fnames - pred_fnames
if missing_in_pred:
    print(f"WARNING: {len(missing_in_pred)} GT images have no pred match — they will be skipped:")
    for f in sorted(missing_in_pred): print(f"  {f}")

print(f"Total images to evaluate: {len(all_fnames)}")

Total images to evaluate: 54


In [41]:
# ============================================================
# STEP 3: Helper functions 
# ============================================================

def polygon_to_mask(segmentation, height, width):
    """Decode a COCO polygon or RLE segmentation to a boolean mask."""
    if isinstance(segmentation, dict):
        # RLE format: {"size": [H, W], "counts": "..."}
        if isinstance(segmentation["counts"], str):
            segmentation["counts"] = segmentation["counts"].encode("utf-8")
        return maskUtils.decode(segmentation).astype(bool)
    else:
        # Polygon format: [[x1, y1, x2, y2, ...]]
        rles = maskUtils.frPyObjects(segmentation, height, width)
        rle  = maskUtils.merge(rles)
        return maskUtils.decode(rle).astype(bool)

def compute_mask_iou(mask1, mask2):
    intersection = np.logical_and(mask1, mask2).sum()
    union = np.logical_or(mask1, mask2).sum()
    return intersection / union if union > 0 else 0.0

def bbox_from_ann(ann):
    """Return [x, y, w, h] from a COCO annotation (uses bbox field if present)."""
    if "bbox" in ann and ann["bbox"]:
        return ann["bbox"]
    return None

def boxes_overlap(b1, b2):
    if b1 is None or b2 is None:
        return True   # can't rule out overlap, compute mask IoU anyway
    x1, y1, w1, h1 = b1
    x2, y2, w2, h2 = b2
    return not (x1 + w1 < x2 or x2 + w2 < x1 or y1 + h1 < y2 or y2 + h2 < y1)


def match_masks(gt_masks, gt_boxes, pred_masks, pred_boxes, iou_threshold):
    """
    Hungarian matching on mask IoU.
    Returns (matched_pairs, unmatched_pred_indices, unmatched_gt_indices).
    """
    if len(gt_masks) == 0 or len(pred_masks) == 0:
        return [], list(range(len(pred_masks))), list(range(len(gt_masks)))

    iou_matrix = np.zeros((len(gt_masks), len(pred_masks)))
    for i, (gm, gb) in enumerate(zip(gt_masks, gt_boxes)):
        for j, (pm, pb) in enumerate(zip(pred_masks, pred_boxes)):
            if boxes_overlap(gb, pb):          # cheap bbox pre-filter
                iou_matrix[i, j] = compute_mask_iou(gm, pm)

    gt_idx, pred_idx = linear_sum_assignment(-iou_matrix)

    matched_pairs, matched_gt, matched_pred = [], set(), set()
    for g, p in zip(gt_idx, pred_idx):
        if iou_matrix[g, p] >= iou_threshold:
            matched_pairs.append((g, p, iou_matrix[g, p]))
            matched_gt.add(g)
            matched_pred.add(p)

    unmatched_pred = [p for p in range(len(pred_masks)) if p not in matched_pred]
    unmatched_gt   = [g for g in range(len(gt_masks))   if g not in matched_gt]
    return matched_pairs, unmatched_pred, unmatched_gt



def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return precision, recall, f1

In [42]:
# ============================================================
# STEP 4: Main evaluation loop
# ============================================================

instance_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
pixel_stats    = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
global_pixel_stats = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

category_stats = defaultdict(lambda: {"tp": 0, "fp": 0})

for fname in all_fnames:
    meta    = image_meta[fname]
    height  = meta["height"]
    width   = meta["width"]
    ecotype = meta["ecotype"]

    gt_anns = gt_anns_by_fname[fname]

    # if len(gt_anns) == 0:
    #     print(f"SKIP!   {ecotype:20s}  GT={len(gt_anns):4d}  ({fname})")
    #     continue

    pred_anns = pred_anns_by_fname[fname]

    gt_masks   = [polygon_to_mask(ann["segmentation"], height, width) for ann in gt_anns]

    if IS_FUSED:
        pred_masks = [polygon_to_mask(ann["segmentation"], height, width) for ann in pred_anns if ann["category_id"] != 0]
    else:
        pred_masks = [polygon_to_mask(ann["segmentation"], height, width) for ann in pred_anns]

    gt_boxes   = [bbox_from_ann(ann) for ann in gt_anns]
    pred_boxes = [bbox_from_ann(ann) for ann in pred_anns]

    matched_pairs, unmatched_pred, unmatched_gt = match_masks(
        gt_masks, gt_boxes, pred_masks, pred_boxes, IOU_THRESHOLD
    )

    # --- Instance-level ---
    tp = len(matched_pairs)
    fp = len(unmatched_pred)
    fn = len(unmatched_gt)
    instance_stats[ecotype]["tp"] += tp
    instance_stats[ecotype]["fp"] += fp
    instance_stats[ecotype]["fn"] += fn


    if IS_FUSED:
        # --- Per-category stats (prediction's category_id only) ---
        for g, p, _ in matched_pairs:
            pred_category = pred_anns[p]["category_id"]
            category_stats[pred_category]["tp"] += 1

        for p in unmatched_pred:
            pred_category = pred_anns[p]["category_id"]
            category_stats[pred_category]["fp"] += 1



    # --- Pixel-level (matched pairs only) ---
    for g, p, _ in matched_pairs:
        gm = gt_masks[g]
        pm = pred_masks[p]
        pixel_stats[ecotype]["tp"] += int(np.logical_and(gm, pm).sum())
        pixel_stats[ecotype]["fp"] += int(np.logical_and(~gm, pm).sum())
        pixel_stats[ecotype]["fn"] += int(np.logical_and(gm, ~pm).sum())

    # --- Global pixel-level (whole-image coverage, no instance matching) ---
    combined_gt   = np.logical_or.reduce(gt_masks)   if gt_masks   else np.zeros((height, width), dtype=bool)
    combined_pred = np.logical_or.reduce(pred_masks) if pred_masks else np.zeros((height, width), dtype=bool)
    global_pixel_stats[ecotype]["tp"] += int(np.logical_and(combined_gt,  combined_pred).sum())
    global_pixel_stats[ecotype]["fp"] += int(np.logical_and(~combined_gt, combined_pred).sum())
    global_pixel_stats[ecotype]["fn"] += int(np.logical_and(combined_gt,  ~combined_pred).sum())

    print(f"{ecotype:20s}  GT={len(gt_anns):4d}  Pred={len(pred_anns):4d}  TP={tp}  FP={fp}  FN={fn}  {fname}")

    
    # visualization overlay
    # img_path = meta.get("file_name", fname)  
    # img = cv2.imread(img_path)
    # img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # overlay = img_rgb.copy()

    # # true positives -- green, pulled from the PREDICTED mask of each matched pair
    # for g, p, _ in matched_pairs:
    #     overlay[pred_masks[p]] = [0, 255, 0]

    # # false positives -- orange, predictions with no matching ground truth
    # for p in unmatched_pred:
    #     overlay[pred_masks[p]] = [255, 165, 0]

    # # false negatives -- red, ground truth instances nothing matched
    # for g in unmatched_gt:
    #     overlay[gt_masks[g]] = [255, 0, 0]

    # plt.figure(figsize=(10, 10))
    # plt.imshow(overlay)
    # plt.axis("off")
    # plt.title(f"{ecotype}: TP={tp} (green)  FP={fp} (orange)  FN={fn} (red)")
    # plt.tight_layout()
    # plt.show()

print("\nDone.")

Pu2-23                GT=  58  Pred=  55  TP=34  FP=20  FN=24  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000001.tif
Pu2-23                GT=  66  Pred=  60  TP=42  FP=17  FN=24  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000002.tif
Pu2-23                GT=  71  Pred=  57  TP=35  FP=21  FN=36  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000003.tif
Ws-0                  GT=  24  Pred=  26  TP=17  FP=8  FN=7  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000008.tif
Ws-0                  GT=  38  Pred=  35  TP=25  FP=9  FN=13  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000009.tif
Ws-0                  GT=  34  Pred=  33  TP=24  FP=8  FN=10  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_1000_L/000010.tif
Shahdara              GT=  27  Pred=  28  TP=21  FP=6  FN=6  /home/mellanie/Desktop/Annotated_dataset/20240918_N4_Plate1_DAG6_10

In [43]:
# print the per catagory metrics if fused (fused imples classifier was used)
if IS_FUSED:
    import csv

    CLASSES = [
        "root",                
        "root_hair",            
        "bump",   
        "background_root_hair", 
        "tangled_root_hair",                 
        "edge_root_hair",                 
        "embedded_root_hair",               
        "bubble",   
        "dirt"    
    ]

    CATEGORY_NAMES = {c["id"]: c["name"] for c in CLASSES} if isinstance(CLASSES[0], dict) else {}

    with open("outputs/category_performance.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["category_id", "category_name", "tp", "fp", "precision"])

        for cat_id in sorted(category_stats.keys()):
            stats = category_stats[cat_id]
            tp, fp = stats["tp"], stats["fp"]
            precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
            cat_name = CATEGORY_NAMES.get(cat_id, f"category_{cat_id}")

            writer.writerow([cat_id, cat_name, tp, fp, f"{precision:.4f}"])

    print("Saved per-category performance to outputs/category_performance.csv")

Saved per-category performance to outputs/category_performance.csv


In [44]:
# ============================================================
# STEP 5 — Print report
# ============================================================
def print_report(stats, title):
    print(f"\n{'='*75}\n{title}\n{'='*75}")
    print(f"{'Ecotype':<25}{'TP':>8}{'FP':>8}{'FN':>8}{'Precision':>12}{'Recall':>10}{'F1':>8}")
    print("-" * 79)
    total_tp = total_fp = total_fn = 0
    for ecotype, s in sorted(stats.items()):
        p, r, f1 = precision_recall_f1(s["tp"], s["fp"], s["fn"])
        print(f"{ecotype:<25}{s['tp']:>8}{s['fp']:>8}{s['fn']:>8}{p:>12.4f}{r:>10.4f}{f1:>8.4f}")
        total_tp += s["tp"]; total_fp += s["fp"]; total_fn += s["fn"]
    mp, mr, mf1 = precision_recall_f1(total_tp, total_fp, total_fn)
    print("-" * 79)
    print(f"{'TOTAL (micro-avg)':<25}{total_tp:>8}{total_fp:>8}{total_fn:>8}{mp:>12.4f}{mr:>10.4f}{mf1:>8.4f}")

print_report(instance_stats,      "INSTANCE-LEVEL EVALUATION")
print_report(pixel_stats,         "PIXEL-LEVEL EVALUATION (matched pairs)")
print_report(global_pixel_stats,  "GLOBAL PIXEL EVALUATION (whole-image coverage)")


INSTANCE-LEVEL EVALUATION
Ecotype                        TP      FP      FN   Precision    Recall      F1
-------------------------------------------------------------------------------
Bay-0                          98      46      58      0.6806    0.6282  0.6533
Col-0*                         57      24      15      0.7037    0.7917  0.7451
Cvi-0                          95      64      88      0.5975    0.5191  0.5556
Ge-0                          117      75     112      0.6094    0.5109  0.5558
Hs-0                           75      35      51      0.6818    0.5952  0.6356
Kondara                       124      54      70      0.6966    0.6392  0.6667
Lip-0                         100     102     110      0.4950    0.4762  0.4854
No-0                           79     123     157      0.3911    0.3347  0.3607
Oy-0                           79      37      41      0.6810    0.6583  0.6695
Pu2-23                        111      58      84      0.6568    0.5692  0.6099
Shahdara     

In [45]:
# ============================================================
# STEP 6 — Save CSV
# ============================================================
rows = []
for ecotype in sorted(instance_stats.keys()):
    inst  = instance_stats[ecotype]
    pix   = pixel_stats[ecotype]
    gpix  = global_pixel_stats[ecotype]

    ip, ir, if1     = precision_recall_f1(inst["tp"],  inst["fp"],  inst["fn"])
    pp, pr, pf1     = precision_recall_f1(pix["tp"],   pix["fp"],   pix["fn"])
    gp, gr, gf1     = precision_recall_f1(gpix["tp"],  gpix["fp"],  gpix["fn"])

    rows.append({
        "genotype":               ecotype,
        "instance_precision":     ip,
        "instance_recall":        ir,
        "instance_f1":            if1,
        "pixel_precision":        pp,
        "pixel_recall":           pr,
        "pixel_f1":               pf1,
        "global_pixel_precision": gp,
        "global_pixel_recall":    gr,
        "global_pixel_f1":        gf1,
    })

df = pd.DataFrame(rows)

# Micro-average totals row
def total_row(stats_dict, prefix):
    tp = sum(s["tp"] for s in stats_dict.values())
    fp = sum(s["fp"] for s in stats_dict.values())
    fn = sum(s["fn"] for s in stats_dict.values())
    p, r, f1 = precision_recall_f1(tp, fp, fn)
    return {f"{prefix}_precision": p, f"{prefix}_recall": r, f"{prefix}_f1": f1}

total = {"genotype": "TOTAL (micro-avg)"}
total.update(total_row(instance_stats,     "instance"))
total.update(total_row(pixel_stats,        "pixel"))
total.update(total_row(global_pixel_stats, "global_pixel"))
df = pd.concat([df, pd.DataFrame([total])], ignore_index=True)

df.to_csv(OUTPUT_CSV, index=False)
print(f"Saved → {OUTPUT_CSV}")
df

Saved → analysis/csv_data_for_analysis/fusion2_100%_yes_tangles.csv


,genotype,instance_precision,instance_recall,instance_f1,pixel_precision,pixel_recall,pixel_f1,global_pixel_precision,global_pixel_recall,global_pixel_f1
0,Bay-0,0.680556,0.628205,0.653333,0.695146,0.911599,0.788793,0.706620,0.814114,0.756568
1,Col-0*,0.703704,0.791667,0.745098,0.704223,0.914745,0.795796,0.668639,0.866617,0.754863
2,Cvi-0,0.597484,0.519126,0.555556,0.686544,0.910955,0.782987,0.693576,0.786165,0.736974
3,Ge-0,0.609375,0.510917,0.555819,0.733617,0.885612,0.802480,0.749762,0.750302,0.750032
4,Hs-0,0.681818,0.595238,0.635593,0.683430,0.930840,0.788176,0.682000,0.868656,0.764094
5,Kondara,0.696629,0.639175,0.666667,0.679691,0.905902,0.776660,0.675715,0.814595,0.738684
6,Lip-0,0.495050,0.476190,0.485437,0.821942,0.792460,0.806932,0.810900,0.672066,0.734985
7,No-0,0.391089,0.334746,0.360731,0.698608,0.858657,0.770408,0.613876,0.675039,0.643007
8,Oy-0,0.681034,0.658333,0.669492,0.695137,0.908530,0.787636,0.708208,0.788517,0.746208
9,Pu2-23,0.656805,0.569231,0.609890,0.764156,0.839061,0.799859,0.783084,0.749340,0.765840
